In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from typing import TypedDict
from langgraph.graph import StateGraph, START

class State(TypedDict):
    topic: str
    joke: str

def refine_topic(state: State):
    return {"topic": state["topic"] + " and cats"}

def generate_joke(state: State):
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile()
)

In [3]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="values",
):
    print(chunk)

{'topic': 'ice cream'}
{'topic': 'ice cream and cats'}
{'topic': 'ice cream and cats', 'joke': 'This is a joke about ice cream and cats'}


In [4]:
for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode="updates",
):
    print(chunk)

{'refine_topic': {'topic': 'ice cream and cats'}}
{'generate_joke': {'joke': 'This is a joke about ice cream and cats'}}


In [5]:
from langgraph.types import StreamWriter

def generate_joke_with_custom(state: State, writer: StreamWriter):
    writer({"custom_key": "Writing custom data while generating a joke"})
    return {"joke": f"This is a joke about {state['topic']}"}

graph_custom = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node("generate_joke", generate_joke_with_custom)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile()
)

for chunk in graph_custom.stream(
    {"topic": "ice cream"},
    stream_mode="custom",
):
    print(chunk)

{'custom_key': 'Writing custom data while generating a joke'}


In [7]:
from dotenv import load_dotenv
import os
# 加载 .env 文件中的环境变量，请确保 .env 文件位于当前工作目录下
load_dotenv()
from langchain_openai import ChatOpenAI


llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE"),
)

def generate_joke_with_llm(state: State):
    llm_response = llm.invoke(
        [{"role": "user", "content": f"Generate a joke about {state['topic']}"}]
    )
    return {"joke": llm_response.content}

graph_with_llm = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node("generate_joke", generate_joke_with_llm)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile()
)

for message_chunk, metadata in graph_with_llm.stream(
    {"topic": "ice cream"},
    stream_mode="messages",
):
    if message_chunk.content:
        print(message_chunk.content, end="|", flush=True)

Why| don|'t| cats| like| ice| cream|?

|Because| they| can|'t| get| past| the| cone|.|

In [8]:
for chunk in graph_custom.stream(
    {"topic": "ice cream"},
    stream_mode="debug",
):
    print(chunk)

{'step': 1, 'timestamp': '2026-04-20T14:40:57.764336+00:00', 'type': 'task', 'payload': {'id': '485acb3e-655d-0a8d-da82-f090fb3577fe', 'name': 'refine_topic', 'input': {'topic': 'ice cream'}, 'triggers': ('branch:to:refine_topic',)}}
{'step': 1, 'timestamp': '2026-04-20T14:40:57.764485+00:00', 'type': 'task_result', 'payload': {'id': '485acb3e-655d-0a8d-da82-f090fb3577fe', 'name': 'refine_topic', 'error': None, 'result': {'topic': 'ice cream and cats'}, 'interrupts': []}}
{'step': 2, 'timestamp': '2026-04-20T14:40:57.764565+00:00', 'type': 'task', 'payload': {'id': '79833787-5a5c-6ea0-b1a0-7e7c269664b4', 'name': 'generate_joke', 'input': {'topic': 'ice cream and cats'}, 'triggers': ('branch:to:generate_joke',)}}
{'step': 2, 'timestamp': '2026-04-20T14:40:57.764667+00:00', 'type': 'task_result', 'payload': {'id': '79833787-5a5c-6ea0-b1a0-7e7c269664b4', 'name': 'generate_joke', 'error': None, 'result': {'joke': 'This is a joke about ice cream and cats'}, 'interrupts': []}}


In [9]:
from langgraph.types import StreamWriter

def generate_joke(state: State, writer: StreamWriter):
    writer({"custom_key": "Writing custom data wHITLe generating a joke"})
    return {"joke": f"This is a joke about {state['topic']}"}

graph = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node(generate_joke)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile()
)

for stream_mode, chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom"],
):
    print(f"Stream mode: {stream_mode}")
    print(chunk)
    print("\n")

Stream mode: updates
{'refine_topic': {'topic': 'ice cream and cats'}}


Stream mode: custom
{'custom_key': 'Writing custom data wHITLe generating a joke'}


Stream mode: updates
{'generate_joke': {'joke': 'This is a joke about ice cream and cats'}}




In [10]:
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
import os
from dotenv import load_dotenv
# 加载 .env 文件中的环境变量，请确保 .env 文件位于当前工作目录下
load_dotenv()

llm = ChatOpenAI(
    model="deepseek-chat",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_API_BASE"),
)

def call_model(state: MessagesState):
    response = llm.invoke(state['messages'])
    return {"messages": response}

workflow = StateGraph(MessagesState)
workflow.add_node(call_model)
workflow.add_edge(START, "call_model")
workflow.add_edge("call_model", END)
app = workflow.compile()

inputs = [{"role": "user", "content": "hi!"}]
async for event in app.astream_events({"messages": inputs}, version="v1"):
    kind = event["event"]
    print(f"{kind}: {event['name']}")

on_chain_start: LangGraph
on_chain_start: call_model
on_chat_model_start: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: ChatOpenAI
on_chat_model_stream: Ch

In [11]:
from langgraph.checkpoint.memory import MemorySaver

# 使用 LLM 版本的笑话生成器，但添加持久化
graph_persistent = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node("generate_joke", generate_joke_with_llm)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile(checkpointer=MemorySaver())  # 启用持久化
)

# 定义线程配置
config = {"configurable": {"thread_id": "my_thread_1"}}

print("第一次运行：")
for chunk in graph_persistent.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="updates",
):
    print(chunk)

print("\n获取最终状态：")
print(graph_persistent.get_state(config).values)

第一次运行：
{'refine_topic': {'topic': 'ice cream and cats'}}
{'generate_joke': {'joke': 'Why did the cat start a business selling ice cream?\n\nBecause he heard the profits were *purr*-fect, and he wanted to make a *mint*!'}}

获取最终状态：
{'topic': 'ice cream and cats', 'joke': 'Why did the cat start a business selling ice cream?\n\nBecause he heard the profits were *purr*-fect, and he wanted to make a *mint*!'}


In [12]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver
from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, MessagesState, START, END
conn = sqlite3.connect("checkpoints.sqlite", check_same_thread=False)
checkpointer = SqliteSaver(conn)

# 使用 LLM 版本的笑话生成器，但添加持久化
graph_persistent = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node("generate_joke", generate_joke_with_llm)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile(checkpointer=checkpointer)  # 启用持久化
)

# 定义线程配置
config = {"configurable": {"thread_id": "my_thread_1"}}

print("第一次运行：")
for chunk in graph_persistent.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="updates",
):
    print(chunk)

print("\n获取最终状态：")
print(graph_persistent.get_state(config).values)

第一次运行：
{'refine_topic': {'topic': 'ice cream and cats'}}
{'generate_joke': {'joke': "Why did the cat open an ice cream shop?\n\nBecause it heard the business was the cat's meow... but it quickly realized it was in a terrible *purr*-sition when all the customers were dogs."}}

获取最终状态：
{'topic': 'ice cream and cats', 'joke': "Why did the cat open an ice cream shop?\n\nBecause it heard the business was the cat's meow... but it quickly realized it was in a terrible *purr*-sition when all the customers were dogs."}


In [13]:
print("状态历史回顾：")
state_history = list(graph_persistent.get_state_history(config))
for i, snapshot in enumerate(state_history):
    print(f"\n存档点 {i + 1}:")
    print(f"  ID: {snapshot.config['configurable']['checkpoint_id'][:8]}...")
    print(f"  步骤: {snapshot.metadata}")
    print(f"  状态: {snapshot.values}")
    print(f"  下一个节点: {snapshot.next}")
    print("-" * 50)

状态历史回顾：

存档点 1:
  ID: 1f13cc6f...
  步骤: {'source': 'loop', 'step': 2, 'parents': {}}
  状态: {'topic': 'ice cream and cats', 'joke': "Why did the cat open an ice cream shop?\n\nBecause it heard the business was the cat's meow... but it quickly realized it was in a terrible *purr*-sition when all the customers were dogs."}
  下一个节点: ()
--------------------------------------------------

存档点 2:
  ID: 1f13cc6f...
  步骤: {'source': 'loop', 'step': 1, 'parents': {}}
  状态: {'topic': 'ice cream and cats'}
  下一个节点: ('generate_joke',)
--------------------------------------------------

存档点 3:
  ID: 1f13cc6f...
  步骤: {'source': 'loop', 'step': 0, 'parents': {}}
  状态: {'topic': 'ice cream'}
  下一个节点: ('refine_topic',)
--------------------------------------------------

存档点 4:
  ID: 1f13cc6f...
  步骤: {'source': 'input', 'step': -1, 'parents': {}}
  状态: {}
  下一个节点: ('__start__',)
--------------------------------------------------


In [15]:
from typing import Literal
from langgraph.types import interrupt, Command

# 定义状态类型
class ApprovalState(TypedDict):
    topic: str
    proposed_action_details: str
    final_result: str

# 定义节点函数
def propose_action(state: ApprovalState) -> ApprovalState:
    """提出一个需要人类审批的操作"""
    return {
        **state,
        "proposed_action_details": f"基于主题 '{state['topic']}' 的操作提议：发送营销邮件给1000个客户"
    }

def human_approval_node(state: ApprovalState) -> Command[Literal["execute_action", "revise_action"]]:
    """获取人类审批的节点"""
    approval_request = interrupt(
        {
            "question": "是否批准执行以下操作？",
            "action_details": state["proposed_action_details"],
            "options": ["approve", "deny"]
        }
    )

    if approval_request.get("user_response") == "approve":
        return Command(goto="execute_action")
    else:
        return Command(goto="revise_action")

def execute_action(state: ApprovalState) -> ApprovalState:
    """执行已批准的操作"""
    return {
        **state,
        "final_result": f"✅ 已执行操作: {state['proposed_action_details']}"
    }

def revise_action(state: ApprovalState) -> ApprovalState:
    """修改被拒绝的操作"""
    return {
        **state, 
        "final_result": f"❌ 操作被拒绝，已修改为: 发送营销邮件给50个目标客户（缩小规模）"
    }
# 构建需要人工审批的图
approval_graph_builder = StateGraph(ApprovalState)

# 添加节点
approval_graph_builder.add_node("propose_action", propose_action)
approval_graph_builder.add_node("human_approval", human_approval_node)
approval_graph_builder.add_node("execute_action", execute_action)
approval_graph_builder.add_node("revise_action", revise_action)

# 添加边
approval_graph_builder.add_edge(START, "propose_action")
approval_graph_builder.add_edge("propose_action", "human_approval")
approval_graph_builder.add_edge("revise_action", "human_approval")  # 修改后再次请求审批

# 编译图（必须包含检查点器以支持中断）
approval_graph = approval_graph_builder.compile(checkpointer=MemorySaver())

In [16]:
# 启动审批流程
config = {"configurable": {"thread_id": "approval_thread"}}

print("=== 第一步：启动智能体，等待人工审批 ===")
try:
    result = approval_graph.invoke(
        {"topic": "产品推广活动"}, 
        config=config
    )
    print("执行完成：", result)
except Exception as e:
    print(f"智能体在等待人工审批处中断: {e}")
    
# 检查当前状态
current_state = approval_graph.get_state(config)
print(f"\n当前状态: {current_state.values}")
print(f"下一个节点: {current_state.next}")
print(f"是否被中断: {current_state.tasks[0] if current_state.tasks else 'No interrupts'}")

=== 第一步：启动智能体，等待人工审批 ===
执行完成： {'topic': '产品推广活动', 'proposed_action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", '__interrupt__': [Interrupt(value={'question': '是否批准执行以下操作？', 'action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'options': ['approve', 'deny']}, id='81bac4b17f99132fc3b8e50af4c8b9ad')]}

当前状态: {'topic': '产品推广活动', 'proposed_action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户"}
下一个节点: ('human_approval',)
是否被中断: PregelTask(id='d7cd6040-0cf7-19e5-9bf7-5d47ec59dfd0', name='human_approval', path=('__pregel_pull', 'human_approval'), error=None, interrupts=(Interrupt(value={'question': '是否批准执行以下操作？', 'action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'options': ['approve', 'deny']}, id='81bac4b17f99132fc3b8e50af4c8b9ad'),), state=None, result=None)


In [17]:
# 模拟人工拒绝审批
print("\n=== 第二步：人工拒绝审批 ===")
try:
    result = approval_graph.invoke(
        Command(resume={"user_response": "deny"}), 
        config=config
    )
    print("拒绝后的执行结果：", result)
except Exception as e:
    print(f"继续等待审批: {e}")
    
# 再次检查状态
current_state = approval_graph.get_state(config)
print(f"\n更新后状态: {current_state.values}")
print(f"下一个节点: {current_state.next}")


=== 第二步：人工拒绝审批 ===
拒绝后的执行结果： {'topic': '产品推广活动', 'proposed_action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'final_result': '❌ 操作被拒绝，已修改为: 发送营销邮件给50个目标客户（缩小规模）', '__interrupt__': [Interrupt(value={'question': '是否批准执行以下操作？', 'action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'options': ['approve', 'deny']}, id='4dfa33d88d81561fb854e5efb9df07af')]}

更新后状态: {'topic': '产品推广活动', 'proposed_action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'final_result': '❌ 操作被拒绝，已修改为: 发送营销邮件给50个目标客户（缩小规模）'}
下一个节点: ('human_approval',)


In [19]:
# 最终批准修改后的操作
print("\n=== 第三步：批准修改后的操作 ===")
final_result = approval_graph.invoke(
    Command(resume={"user_response": "approve"}), 
    config=config
)
print("最终执行结果：", final_result["final_result"])
print(approval_graph.get_state(config))


=== 第三步：批准修改后的操作 ===
最终执行结果： ✅ 已执行操作: 基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户
StateSnapshot(values={'topic': '产品推广活动', 'proposed_action_details': "基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户", 'final_result': "✅ 已执行操作: 基于主题 '产品推广活动' 的操作提议：发送营销邮件给1000个客户"}, next=(), config={'configurable': {'thread_id': 'approval_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f13d2be-e764-6676-8005-628ca0da37a6'}}, metadata={'source': 'loop', 'step': 5, 'parents': {}}, created_at='2026-04-21T02:43:51.881232+00:00', parent_config={'configurable': {'thread_id': 'approval_thread', 'checkpoint_ns': '', 'checkpoint_id': '1f13d2be-e761-6b7e-8004-dbcb2d1fd007'}}, tasks=(), interrupts=())


In [21]:
from typing import Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.types import interrupt, Command
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage

# 定义完整的状态类
class HumanInLoopState(TypedDict):
    user_question: str
    generated_summary: str
    messages: Annotated[list, add_messages]

def summary_generation_node(state: HumanInLoopState):
    """生成初始摘要的节点"""
    user_question = state["user_question"]

    # 模拟LLM生成摘要
    generated_summary = f"""Based on the question "{user_question}", 
here's an initial summary:
    
LangGraph is a framework for building AI agents with the following 
features:
- State management capabilities
- Graph-based workflow design
- Integration with LangChain ecosystem
- Support for complex agent interactions

This summary needs human review for accuracy and completeness."""

    return {
        "generated_summary": generated_summary,
        "messages": [AIMessage(content=f"Generated initial summary for: {user_question}")]
    }

def human_review_node(state: HumanInLoopState):
    """人工审核节点 - 使用 Interrupt 请求人工干预"""
    current_summary = state["generated_summary"]

    # 触发中断，请求人工审核
    edited_summary = interrupt(
        {
            "task": "Please review and edit the generated summary",
            "current_summary": current_summary,
            "instructions": "Improve accuracy, add missing details, and ensure clarity"
        }
    )

    return {
        "generated_summary": edited_summary,
        "messages": [AIMessage(content="Summary sent for human review")]
    }

# 创建状态图
graph_builder = StateGraph(HumanInLoopState)

# 添加节点
graph_builder.add_node("generate_summary", summary_generation_node)
graph_builder.add_node("human_review", human_review_node)

# 定义边
graph_builder.add_edge(START, "generate_summary")
graph_builder.add_edge("generate_summary", "human_review")
graph_builder.add_edge("human_review", END)

# 编译图 - 在人工审核节点前中断
memory_saver = MemorySaver()
complete_human_loop_graph = graph_builder.compile(
    checkpointer=memory_saver,
)

In [22]:
# 💡 第一步：运行图直到中断点
thread_config = {"configurable": {"thread_id": "complete_human_loop"}}

print("🚀 第一步：运行图直到人工审核节点...")
print("=" * 50)

# 初始运行 - 会在 human_review 节点前停止
first_result = complete_human_loop_graph.invoke(
    {
        "user_question": "What are the key advantages of LangGraph for enterprise AI applications?",
        "messages": [HumanMessage(content="Please generate a comprehensive summary")]
    },
    config=thread_config
)

print(f"📊 第一步结果: {first_result}")
print()

# 检查当前图状态
current_state = complete_human_loop_graph.get_state(thread_config)
print(f"🔍 当前状态值: {current_state.values}")
print(f"⏭️ 下一个待执行节点: {current_state.next}")
print(f"📋 待处理任务: {current_state.tasks}")

if current_state.tasks:
    task_info = current_state.tasks[0]
    print(f"🎯 任务详情: {task_info}")
    if hasattr(task_info, 'interrupts') and task_info.interrupts:
        interrupt_data = task_info.interrupts[0]
        print(f"⚠️ 中断数据: {interrupt_data}")
        if hasattr(interrupt_data, 'value'):
            print(f"📝 需要审核的内容: {interrupt_data.value}")

🚀 第一步：运行图直到人工审核节点...
📊 第一步结果: {'user_question': 'What are the key advantages of LangGraph for enterprise AI applications?', 'generated_summary': 'Based on the question "What are the key advantages of LangGraph for enterprise AI applications?", \nhere\'s an initial summary:\n\nLangGraph is a framework for building AI agents with the following \nfeatures:\n- State management capabilities\n- Graph-based workflow design\n- Integration with LangChain ecosystem\n- Support for complex agent interactions\n\nThis summary needs human review for accuracy and completeness.', 'messages': [HumanMessage(content='Please generate a comprehensive summary', additional_kwargs={}, response_metadata={}, id='89e364b4-7e03-417f-9ed4-bf338a4c4a30'), AIMessage(content='Generated initial summary for: What are the key advantages of LangGraph for enterprise AI applications?', additional_kwargs={}, response_metadata={}, id='e7dfd5a2-0b1c-4ae1-a116-29332ac56790')], '__interrupt__': [Interrupt(value={'task': 'Please 

In [23]:
# 💡 第二步：模拟人工审核并提供改进的摘要
print("\n🚀 第二步：提供人工审核后的改进摘要...")
print("=" * 50)

# 模拟人工审核后的改进摘要
improved_summary = """LangGraph provides significant advantages for 
enterprise AI applications:

🏗️ **Enterprise-Grade Architecture**:
- Robust state management with built-in persistence
- Scalable graph-based workflow orchestration
- Production-ready error handling and recovery

🔧 **Advanced Integration Capabilities**:
- Seamless integration with LangChain ecosystem
- Support for multiple LLM providers and tools
- Human-in-the-loop workflows for quality assurance

⚡ **Performance & User Experience**:
- Real-time streaming capabilities for responsive interfaces
- Efficient checkpoint and resume functionality
- Memory management for long-running conversations

🛡️ **Enterprise Features**:
- Comprehensive monitoring and observability
- Security controls and access management
- Deployment flexibility across cloud and on-premise

This makes LangGraph an ideal choice for mission-critical enterprise AI 
agent deployments."""

print(f"✏️ 人工改进的摘要:\n{improved_summary}")
print(f"\n📏 摘要长度: {len(improved_summary)} 字符")


🚀 第二步：提供人工审核后的改进摘要...
✏️ 人工改进的摘要:
LangGraph provides significant advantages for 
enterprise AI applications:

🏗️ **Enterprise-Grade Architecture**:
- Robust state management with built-in persistence
- Scalable graph-based workflow orchestration
- Production-ready error handling and recovery

🔧 **Advanced Integration Capabilities**:
- Seamless integration with LangChain ecosystem
- Support for multiple LLM providers and tools
- Human-in-the-loop workflows for quality assurance

⚡ **Performance & User Experience**:
- Real-time streaming capabilities for responsive interfaces
- Efficient checkpoint and resume functionality
- Memory management for long-running conversations

🛡️ **Enterprise Features**:
- Comprehensive monitoring and observability
- Security controls and access management
- Deployment flexibility across cloud and on-premise

This makes LangGraph an ideal choice for mission-critical enterprise AI 
agent deployments.

📏 摘要长度: 907 字符


In [24]:
# 💡 第三步：使用 Command 恢复图执行，传入人工审核的结果
print("🚀 第三步：恢复图执行，使用人工审核的摘要...")
print("=" * 50)

# 使用 Command 恢复图执行，传入改进后的摘要
final_result = complete_human_loop_graph.invoke(
    Command(resume=improved_summary),  # 将改进的摘要作为中断节点的返回值
    config=thread_config
)

print(f"✅ 最终执行结果: {final_result}")
print()

# 获取最终状态
final_state = complete_human_loop_graph.get_state(thread_config)
print(f"🎯 最终状态: {final_state.values}")
print(f"📝 最终摘要长度: {len(final_result.get('generated_summary', ''))} 字符")
print()

# 显示消息历史
print("📨 消息历史:")
for i, msg in enumerate(final_result.get('messages', []), 1):
    print(f"  {i}. {msg.content}")

print(f"\n🏁 人工在环流程完成！摘要已通过人工审核并改进。")

🚀 第三步：恢复图执行，使用人工审核的摘要...
✅ 最终执行结果: {'user_question': 'What are the key advantages of LangGraph for enterprise AI applications?', 'generated_summary': 'LangGraph provides significant advantages for \nenterprise AI applications:\n\n🏗️ **Enterprise-Grade Architecture**:\n- Robust state management with built-in persistence\n- Scalable graph-based workflow orchestration\n- Production-ready error handling and recovery\n\n🔧 **Advanced Integration Capabilities**:\n- Seamless integration with LangChain ecosystem\n- Support for multiple LLM providers and tools\n- Human-in-the-loop workflows for quality assurance\n\n⚡ **Performance & User Experience**:\n- Real-time streaming capabilities for responsive interfaces\n- Efficient checkpoint and resume functionality\n- Memory management for long-running conversations\n\n🛡️ **Enterprise Features**:\n- Comprehensive monitoring and observability\n- Security controls and access management\n- Deployment flexibility across cloud and on-premise\n\nThis makes